# 信噪 · Qwen2.5-1.5B LoRA 微调 (Kaggle)

把 50 条手工种子 + LLM 扩写得到的 ~250 条 SFT 数据微调到 Qwen2.5-1.5B-Instruct 上。

## 准备工作
1. **上传训练数据**: 把 `data/sft_train.jsonl` 和 `data/sft_val.jsonl` 打包成 Kaggle Dataset，加入到本 notebook 的 input。建议数据集名字 `xinzao-sft-data`。
2. **GPU**: 右侧 Settings → Accelerator → 选 GPU T4 x2 (推荐) 或 P100。
3. **Internet**: 右侧 Settings → 打开 Internet（要下 Qwen 模型）。

训练约 30-60 分钟，最终输出 LoRA adapter (~25MB)，可下载到本地或推到 HF Hub。

## 1. 安装依赖

Unsloth 在 Kaggle T4 上能比原生 trl 快 2-3 倍、显存省 30%，强烈推荐。
若安装出问题，把 unsloth 那行注释掉，改用纯 trl + peft（训练慢一点但稳）。

In [ ]:
# Unsloth (推荐，加速 2-3x)
!pip install -q -U "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
# 训练框架
!pip install -q -U trl peft accelerate datasets transformers

In [ ]:
import os, json, torch
from pathlib import Path

# 找到 Kaggle Dataset 里上传的 jsonl
INPUT_ROOT = Path("/kaggle/input")
candidate_files = list(INPUT_ROOT.rglob("sft_train.jsonl"))
assert candidate_files, "找不到 sft_train.jsonl，请先上传为 Kaggle Dataset 并加入 input"
TRAIN_PATH = candidate_files[0]
VAL_PATH = TRAIN_PATH.parent / "sft_val.jsonl"
print(f"train: {TRAIN_PATH}")
print(f"val:   {VAL_PATH}")

OUTPUT_DIR = Path("/kaggle/working/qwen-1.5b-xinzao-lora")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"output: {OUTPUT_DIR}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU'}")

## 2. 加载 Qwen2.5-1.5B + 配置 LoRA

In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen2.5-1.5B-Instruct"   # 同一个模型，但 Unsloth 预静态量化加载更快
MAX_SEQ_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,            # auto: bfloat16 如果 GPU 支持
    load_in_4bit=False,    # 1.5B 不需要 4-bit 量化，T4 显存够
)

# 给模型装上 LoRA adapter
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                                 # LoRA rank
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",  # 节省 30% 显存
    random_state=42,
)
model.print_trainable_parameters()

## 3. 加载数据 + 应用 chat template

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={
        "train": str(TRAIN_PATH),
        "validation": str(VAL_PATH),
    },
)
print(dataset)
print("\n--- 示例样本 ---")
print(json.dumps(dataset['train'][0]['messages'], ensure_ascii=False, indent=2))

In [ ]:
def apply_chat_template(example):
    """把 messages 转成 Qwen 期望的单一文本字段。"""
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

dataset = dataset.map(apply_chat_template, remove_columns=dataset["train"].column_names)
print(dataset)
print("\n--- 应用 chat template 后的示例 ---")
print(dataset['train'][0]['text'][:500])

## 4. 配置 SFTTrainer + 训练

**关键超参**：
- batch_size=4, grad_accum=4 → effective batch 16，T4 显存友好
- 3 epochs，~250 样本 × 3 = 750 step（实际经 grad_accum 是 ~47 优化器 step）
- lr=2e-4 cosine，warmup 10%
- weight_decay=0.01 防过拟合

In [ ]:
from trl import SFTConfig, SFTTrainer

config = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=2,
    eval_strategy="steps",
    eval_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    bf16=True,
    optim="adamw_8bit",                   # bitsandbytes 8-bit Adam 节省显存
    seed=42,
    max_seq_length=MAX_SEQ_LEN,
    dataset_text_field="text",
    packing=False,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=config,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
)

train_result = trainer.train()
print("\n--- 训练完成 ---")
print(train_result.metrics)

## 5. 保存 LoRA adapter

In [ ]:
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

import subprocess
subprocess.run(["ls", "-lh", str(OUTPUT_DIR)])
print(f"\n作为 Kaggle Notebook Output 保存：/kaggle/working/qwen-1.5b-xinzao-lora")
print(f"右侧 Notebook 运行后能从 Output 面板下载整个文件夹。")

## 6. 快速推理测试

训练后抽几个场景试下效果。

In [ ]:
FastLanguageModel.for_inference(model)

test_inputs = [
    "我刚收到一条短信：【工商银行】您的账户存在风险，请立即登录 http://icbc-secure.cn.vip 验证身份。是真的吗？",
    "我妈接到电话说是检察院的要她把钱转到安全账户怎么办？",
    "我所有账号都用 password123，安全吗？",
]

for q in test_inputs:
    msgs = [
        {"role": "system", "content": "你是信噪，反诈顾问。语气锋利但不油腻，三步以内行动建议。"},
        {"role": "user", "content": q},
    ]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.7,
        do_sample=True,
        top_p=0.9,
    )
    reply = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f"问：{q}")
    print(f"信噪：{reply}")
    print("-" * 80)

## 7. (可选) 推送到 Hugging Face Hub

如果你有 HF 账号且设了 `HF_TOKEN` Secret，可以直接 push。未设跳过本 cell。

In [ ]:
# from kaggle_secrets import UserSecretsClient
# token = UserSecretsClient().get_secret("HF_TOKEN")
# model.push_to_hub("YOUR_HF_USERNAME/qwen-1.5b-xinzao-lora", token=token)
# tokenizer.push_to_hub("YOUR_HF_USERNAME/qwen-1.5b-xinzao-lora", token=token)
# print("Pushed to HF Hub")